In [ ]:
"""
Repro.ipynb - Simplified version for testing DeepSeek only across 1, 2, 3-back with 1 block each.

Automatically generated by Colab.
"""

from google.colab import drive
drive.mount('/content/drive')

WORKING_DIR = '/content/drive/MyDrive/FTEC5560'
import os
os.chdir(WORKING_DIR)
print(f"Changed working directory to: {os.getcwd()}")

# Install Libraries (Only once per session if not already installed) ---
# !pip install openai numpy pandas scipy matplotlib seaborn

# Import Libraries, Define Configuration, and Helper Functions ---

import random
import numpy as np
import pandas as pd
from openai import OpenAI
from scipy.stats import norm
import os
import glob
from google.colab import userdata
import time

# Retrieve the API Key from Colab Secrets for security
API_KEY_ENV = userdata.get('siliconflow_api_key')
BASE_URL = "https://api.siliconflow.cn/v1"

# Define the models, N-back difficulties, and temperature parameters to test
MODELS_TO_TEST = [
    "deepseek-ai/DeepSeek-V3.2", # Using DeepSeek only as requested
]
# Test 1, 2, and 3-back
N_BACK_VALUES = [1, 2, 3]
TEMPERATURES_TO_TEST = [0.7]
NUM_BLOCKS_PER_N = 1 # Run only 1 block per N-back condition

# Specify the paths to the dataset folders
DATASET_PATHS = {
    1: "1back/",
    2: "2back/",
    3: "3back/", # Assumes a '3back/' folder exists
    # Add more if needed, e.g., 4: "4back/"
}

# Define the NEW number of trials per block based on your actual dataset
NEW_TRIALS_PER_BLOCK = 24

def load_sequence_and_targets_from_file(file_path):
    """Load the letter sequence and target labels from a specified file."""
    with open(file_path, 'r') as f:
        content = f.read().strip()

    split_point = NEW_TRIALS_PER_BLOCK
    sequence_part = content[:split_point]
    targets_part = content[split_point:].strip()

    if len(sequence_part) != NEW_TRIALS_PER_BLOCK or len(targets_part) != NEW_TRIALS_PER_BLOCK:
        print(f"ERROR: File {file_path} does not contain exactly {NEW_TRIALS_PER_BLOCK} letters and {NEW_TRIALS_PER_BLOCK} targets after processing.")
        print(f"       Got {len(sequence_part)} letters and {len(targets_part)} targets.")
        raise ValueError(f"Incorrect file format for {file_path}")

    sequence = list(sequence_part.upper())
    targets = list(targets_part.lower())
    return sequence, targets

def get_model_response(client, full_conversation_history, model_name, temperature=0.7):
    """
    Call the model API to get a response using the full conversation history.
    """
    start_time = time.time()

    completion = client.chat.completions.create(
        model=model_name,
        messages=full_conversation_history, # Pass the entire history
        max_tokens=10,
        temperature=temperature,
    )

    response_time = time.time() - start_time

    response_text = completion.choices[0].message.content.strip().lower()
    print(f"      Raw API Response: '{response_text}', RT: {response_time:.2f}s")

    if not response_text:
        print("        WARNING: Empty response received.")
        return '-', response_time

    first_char = response_text[0]
    if first_char == 'm':
        return 'm', response_time
    elif first_char in ['-', 'n']: # Treat 'n' like '-' as an incorrect answer for '-' target
        return '-', response_time
    else:
        print(f"        Rule violation! Extracting the first letter of the response: '{first_char}'")
        extracted_first = first_char
        if extracted_first == 'm':
            return 'm', response_time
        else: # Default to '-' for any other character
             print(f"        Unexpected extracted char '{extracted_first}', defaulting to '-'")
             return '-', response_time

def calculate_metrics_per_block(targets, responses):
    """Calculate psychophysical metrics for a single block based on targets and responses."""
    if len(targets) != len(responses):
        print(f"DEBUG: calculate_metrics_per_block called with mismatch. Targets: {len(targets)}, Responses: {len(responses)}")
        raise ValueError("Targets and responses must have the same length.")

    hits = sum(1 for t, r in zip(targets, responses) if t == 'm' and r == 'm')
    misses = sum(1 for t, r in zip(targets, responses) if t == 'm' and r == '-')
    fas = sum(1 for t, r in zip(targets, responses) if t == '-' and r == 'm')
    crs = sum(1 for t, r in zip(targets, responses) if t == '-' and r == '-')

    total_signal = hits + misses
    total_noise = fas + crs

    hit_rate = hits / total_signal if total_signal > 0 else 0.0
    fa_rate = fas / total_noise if total_noise > 0 else 0.0

    accuracy = (hits + crs) / len(targets) if targets else 0.0

    epsilon = 1e-9
    corrected_hr = np.clip(hit_rate, epsilon, 1 - epsilon)
    corrected_far = np.clip(fa_rate, epsilon, 1 - epsilon)
    d_prime = norm.ppf(corrected_hr) - norm.ppf(corrected_far)

    return {
        'hit_rate': hit_rate, 'fa_rate': fa_rate,
        'accuracy': accuracy, 'd_prime': d_prime,
        'hits': hits, 'misses': misses, 'fas': fas, 'crs': crs
    }

def run_single_condition(n_back, num_blocks, client, model_name, temp_value, condition_label):
    """Run a single experimental condition (model, N-back, temp) for a specified number of blocks."""
    dataset_path = DATASET_PATHS.get(n_back)
    if not dataset_path or not os.path.isdir(dataset_path):
        print(f"WARNING: Dataset path for {n_back}-back ({dataset_path}) does not exist. Skipping condition {condition_label}.")
        return pd.DataFrame() # Return empty DataFrame

    results = []
    all_files = sorted(glob.glob(os.path.join(dataset_path, "*.txt")))
    selected_files = all_files[:num_blocks]

    if not selected_files:
        print(f"WARNING: No .txt files found in {dataset_path}. Skipping condition {condition_label}.")
        return pd.DataFrame()

    print(f"Starting {condition_label}...")
    for idx, file_path in enumerate(selected_files):
        block_id = int(os.path.basename(file_path).split('.')[0])
        print(f"  Processing Block {idx+1}/{num_blocks} from {file_path.split('/')[-1]}...")

        sequence, targets = load_sequence_and_targets_from_file(file_path)
        responses = []
        response_times = []

        # System prompt changes based on N-back value
        system_prompt = f"You are asked to perform a {n_back}-back task. You will see a sequence of letters. Your task is to respond with 'm' (no quotation marks, just the letter m) whenever the current letter is the same as the one {n_back} position(s) back in the sequence, and '-' (no quotation marks, just the dash sign) otherwise. Only 'm' and '-' are allowed responses. No explanations needed: please don't output any extra words!! The sequence will be presented one letter at a time. Now begins the task."

        # Initialize the full conversation history for this block
        full_conversation_history = [
            {"role": "system", "content": system_prompt}
        ]

        for i in range(len(sequence)):
            current_letter = sequence[i]

            # Add the current user input (the letter) to the history
            full_conversation_history.append({"role": "user", "content": current_letter})

            # Get the model's response based on the full history
            try:
                resp, rt = get_model_response(client, full_conversation_history, model_name, temperature=temp_value)
                responses.append(resp)
                response_times.append(rt)

                # Add the model's response (assistant message) to the history for the next iteration
                full_conversation_history.append({"role": "assistant", "content": resp})

            except Exception as e:
                print(f"General Error on trial {i} (stimulus '{current_letter}'): {type(e).__name__} - {e}")
                responses.append('-')
                response_times.append(np.nan)
                # Add placeholder for consistency
                full_conversation_history.append({"role": "assistant", "content": "-"})
                time.sleep(1) # Brief pause on error
                continue

            print(f"        Trial {i}: Current Letter = {current_letter}, Target = {targets[i]}, Response = {resp}, Correct = {'Yes' if resp == targets[i] else 'No'}")

        # Calculate metrics for this specific block
        if len(responses) != len(targets):
             print(f"Warning: Length mismatch for block {block_id}. Targets: {len(targets)}, Responses: {len(responses)}. Skipping block.")
             continue # Skip this block if there's still a mismatch

        block_metrics = calculate_metrics_per_block(targets, responses)
        block_metrics['block_id'] = block_id
        block_metrics['model'] = model_name
        block_metrics['n_back'] = n_back
        block_metrics['temperature'] = temp_value
        block_metrics['avg_rt'] = np.mean([t for t in response_times if not np.isnan(t)])

        results.append(block_metrics)
        print(f"    Completed Block {block_id}. Accuracy: {block_metrics['accuracy']:.3f}, d': {block_metrics['d_prime']:.3f}")

    print(f"Completed {condition_label}. Processed {len(results)} blocks.\n")
    return pd.DataFrame(results)

# Initialize the OpenAI client
client = OpenAI(api_key=API_KEY_ENV, base_url=BASE_URL)

# --- Cell 4: Main Experiment Runner & Results Saving ---
print("--- Starting Main Experiment ---")
all_results_df = pd.DataFrame()

for model_name in MODELS_TO_TEST:
    for n_val in N_BACK_VALUES:
        for temp_val in TEMPERATURES_TO_TEST:
            condition_label = f"{model_name.split('/')[-1]}_Fixed_T{temp_val}_{n_val}back"
            df = run_single_condition(
                n_back=n_val,
                num_blocks=NUM_BLOCKS_PER_N,
                client=client,
                model_name=model_name,
                temp_value=temp_val,
                condition_label=condition_label
            )
            all_results_df = pd.concat([all_results_df, df], ignore_index=True)

# Save the final aggregated results to a CSV file
output_filename = "llm_nback_results_deepseek_only.csv"
all_results_df.to_csv(output_filename, index=False)
print(f"Final DataFrame shape: {all_results_df.shape}")
print(all_results_df)
print(f"\nResults saved to {output_filename}")

# --- Cell 5: Basic Summary Print ---
if not all_results_df.empty and 'n_back' in all_results_df.columns and 'accuracy' in all_results_df.columns:
    print("\n--- Summary by N-back Condition ---")
    summary = all_results_df.groupby('n_back')[['accuracy', 'd_prime', 'avg_rt']].mean().round(4)
    print(summary)
else:
    print("\nNo results to summarize.")

Mounted at /content/drive
Changed working directory to: /content/drive/MyDrive/FTEC5560
--- Starting Main Experiment ---
Starting DeepSeek-V3.2_Fixed_T0.7_1back...
  Processing Block 1/1 from 0.txt...
      Raw API Response: '-', RT: 2.67s
        Trial 0: Current Letter = Q, Target = -, Response = -, Correct = Yes
      Raw API Response: '-', RT: 1.15s
        Trial 1: Current Letter = V, Target = -, Response = -, Correct = Yes
      Raw API Response: '-', RT: 2.43s
        Trial 2: Current Letter = B, Target = -, Response = -, Correct = Yes
      Raw API Response: '-', RT: 1.13s
        Trial 3: Current Letter = V, Target = -, Response = -, Correct = Yes
      Raw API Response: 'm', RT: 1.02s
        Trial 4: Current Letter = V, Target = m, Response = m, Correct = Yes
      Raw API Response: 'm', RT: 0.90s
        Trial 5: Current Letter = V, Target = m, Response = m, Correct = Yes
      Raw API Response: '-', RT: 2.95s
        Trial 6: Current Letter = F, Target = -, Response = -, C